# Per-UID LSTM for IEEE-CIS Fraud — PyTorch version

This notebook mirrors `Time_Series_LSTM_per_UID.ipynb` (Keras) but is written in
PyTorch, following the structural pattern from the Kaggle reference:

> [arunmohan003 — *Sentiment analysis using LSTM - PyTorch*](https://www.kaggle.com/code/arunmohan003/sentiment-analysis-using-lstm-pytorch)

Same upstream logic as the Keras notebook:

- Load parquet checkpoints, using `X_train_copy5.parquet` so UID aggregation columns and `outsider15` are available.
- Bridge train+test before windowing so test rows can use train history.
- Add three time-gap features per UID.
- Standardize (fit on train rows only).
- Build per-UID sliding windows of length `WINDOW`.
- Train under **strict expanding-window time validation** with `MIN_TRAIN_MONTHS = 3`.
- Run rolling temporal ablations for LSTM, temporal CNN, and feature-axis CNN + LSTM using the rebuilt UID-aggregation sequence tensor.

## What's adapted from the sentiment-analysis reference

| arunmohan003's notebook | This notebook |
|--|--|
| Word indices `(batch, seq_len)` → `nn.Embedding` → `(batch, seq_len, embed_dim)` | Numeric features `(batch, seq_len, n_features)` directly into LSTM (no embedding) |
| `vocab_size`, `embedding_dim` hyperparameters | `n_features` only — no vocab |
| `nn.LSTM(input_size=embedding_dim, ...)` | `nn.LSTM(input_size=n_features, ...)` |
| `model.init_hidden(batch_size)` per iteration | Same pattern, kept for fidelity |
| `nn.BCELoss` after sigmoid | Same |
| Manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `clip_grad_norm_`, `optimizer.step()` | Same |
| Best model saved by validation loss | Best model saved by **validation AUC** (better metric for fraud) |

The reason there's no embedding layer: in sentiment analysis each word is a discrete
token that has to be turned into a continuous vector. Your fraud features are already
continuous after `StandardScaler` (and previously-categorical fields like `card1` were
already integer-encoded by the upstream pipeline), so the LSTM can ingest them directly.


## MPS / Apple Silicon GPU notes

PyTorch supports Apple Silicon GPU via the **MPS** (Metal Performance Shaders) backend.
The config cell below picks the best available device automatically: `mps` if you're on
M-series, else `cuda`, else `cpu`.

A few specifics for MPS:

- Some ops fall back to CPU silently. For LSTM this works but you may see warnings
  about unsupported dtypes — mostly harmless.
- `torch.compile(...)` doesn't help much on MPS yet (Metal backend is limited),
  so this notebook doesn't use it.
- Mixed-precision (`autocast`) is supported but not used here — fp32 is more stable
  for LSTM on MPS, and the speed difference is small.
- `num_workers > 0` in `DataLoader` can deadlock on macOS in Jupyter. We use
  `num_workers=0` and rely on the unified-memory architecture for fast host-device
  transfer.


In [ ]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

In [ ]:
# 0. Imports and config — PyTorch + MPS-aware
import os, gc, math, time, datetime, warnings, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- Configuration -----
DATA_DIR = '/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/pkl_exported_files'

WINDOW              = 20      # was 5  (change F)
MIN_TRAIN_MONTHS    = 3
BATCH               = 1024
EPOCHS              = 30      # was 12 (change I)
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0     # was 0.5 (change D)
EARLY_STOP_PATIENCE = 6
SEED                = 42

HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
N_SEEDS      = 3              # for seed ensembling (change H)
USE_POS_WEIGHT = False        # plain BCE for AUC (change C)
USE_STATIC_TOWER = False       # dual-tower (change B)

# ----- Reproducibility -----
torch.manual_seed(SEED); np.random.seed(SEED)

# ----- Device selection -----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  device={device}')


In [ ]:
x = torch.randn(1024, 5, 250).to(device)
print(x.device)

## 1. Load preprocessed parquet checkpoints with UID aggregations


In [ ]:
from pathlib import Path

DATA_DIR_PARQUET = Path("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/parquet_exported_files")

X_TRAIN_PARQUET = DATA_DIR_PARQUET / "X_train_copy5.parquet"  # includes UID aggregations + outsider15
X_TEST_PARQUET  = DATA_DIR_PARQUET / "X_test_copy4.parquet"   # no copy5 test file is currently available
Y_TRAIN_PARQUET = DATA_DIR_PARQUET / "y_train.parquet"

X_train = pd.read_parquet(X_TRAIN_PARQUET)
X_test  = pd.read_parquet(X_TEST_PARQUET)
y_train = pd.read_parquet(Y_TRAIN_PARQUET)["isFraud"].astype("int8")

X_train.columns = pd.Index(np.asarray(X_train.columns, dtype=object), dtype=object)
X_test.columns = pd.Index(np.asarray(X_test.columns, dtype=object), dtype=object)

uid_aggregation_cols = [
    c for c in X_train.columns
    if (
        c == "uid_FE"
        or "_uid_" in c
        or c.startswith("uid_")
        or c.endswith("_uid_mean")
        or c.endswith("_uid_std")
    )
    and c not in {"uid", "uid_v2", "uid_v2_FE"}
]

missing_in_test = [c for c in X_train.columns if c not in X_test.columns]
if missing_in_test:
    print(f"Adding {len(missing_in_test)} train-only columns to X_test as NaN placeholders.")
    for c in missing_in_test:
        X_test[c] = np.nan

extra_test_cols = [c for c in X_test.columns if c not in X_train.columns]
if extra_test_cols:
    print(f"Dropping {len(extra_test_cols)} test-only columns to align with train.")
X_test = X_test.reindex(columns=X_train.columns)

print("train rows :", len(X_train))
print("test  rows :", len(X_test))
print("train cols :", X_train.shape[1])
print("test  cols :", X_test.shape[1])
print("positive rate:", round(float(y_train.mean()), 4))
print("UID aggregation columns in train:", len(uid_aggregation_cols))
print(uid_aggregation_cols)
print("outsider15 in train:", "outsider15" in X_train.columns)


In [ ]:
import pyarrow
print(pyarrow.__version__)

In [ ]:
print("Parquet sources")
print("  train:", X_TRAIN_PARQUET)
print("  test :", X_TEST_PARQUET)
print("  y    :", Y_TRAIN_PARQUET)

assert "outsider15" in X_train.columns, "X_train_copy5.parquet should contain outsider15"
assert len(uid_aggregation_cols) > 0, "No UID aggregation columns detected"

uid_aggs_missing_from_features = [c for c in uid_aggregation_cols if c not in X_train.columns]
print("UID aggregation columns missing from X_train:", uid_aggs_missing_from_features)


In [ ]:
X_train

In [ ]:
X_test

In [ ]:
y_train

## 2. Feature selection


In [ ]:
HOUSEKEEPING = {
    'TransactionDT', 'D6','D7','D8','D9','D12','D13','D14',
    'uid', 'uid_v2', 'uid_v2_FE', 'day', 'DT', 'isFraud',
    'C3','M5','id_08','id_33',
    'card4','id_07','id_14','id_21','id_30','id_32','id_34',
    *(f'id_{x}' for x in range(22, 28)),
    'D1n', 'D3n', 'D2n', 'D5n', 'D9n',
    'DT_M', 'DT_W', 'DT_D', 'DT_M_total', 'DT_W_total', 'DT_D_total'
}

feature_cols = [c for c in X_train.columns if c not in HOUSEKEEPING]
uid_agg_feature_cols = [c for c in uid_aggregation_cols if c in feature_cols]
missing_uid_agg_features = [c for c in uid_aggregation_cols if c not in feature_cols]

print('feature columns:', len(feature_cols))
print('Removed columns: ', len(HOUSEKEEPING))
print('UID aggregation columns retained:', len(uid_agg_feature_cols))
print(uid_agg_feature_cols)
print('UID aggregation columns removed:', missing_uid_agg_features)
print('outsider15 retained:', 'outsider15' in feature_cols)

In [ ]:
missing_housekeeping = sorted(HOUSEKEEPING - set(X_train.columns))
present_housekeeping = sorted(HOUSEKEEPING & set(X_train.columns))

print("HOUSEKEEPING total:", len(HOUSEKEEPING))
print("Present in X_train:", len(present_housekeeping))
print("Missing from X_train:", len(missing_housekeeping))
print(missing_housekeeping)

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

## 3. Bridge train+test, then add time-gap features

Same logic as the Keras notebook: concat first, compute gap features on the combined
frame so test rows of known UIDs can reference their train history.


In [ ]:
X_train['__source__'] = 'train'
X_test['__source__']  = 'test'
X_all = pd.concat([X_train, X_test], axis=0)
# X_all.drop('isFraud', axis=1, inplace=True)
print(f'X_all rows: {len(X_all):,}')

def add_time_gap_features(df):
    df = df.sort_values(['uid', 'DT'], kind='mergesort').copy()
    g = df.groupby('uid', sort=False)

    delta = g['TransactionDT'].diff().fillna(0).astype('float32')
    df['delta_seconds_prev'] = delta
    # df['delta_log_prev']     = np.log1p(delta).astype('float32')
    df['uid_count_so_far']   = g.cumcount().astype('float32') + 1

    prev_amt = g['TransactionAmt'].shift(1)

    df['uid_prev_amt']       = prev_amt.fillna(-1).astype('float32')
    df['uid_amt_diff_prev']  = (df['TransactionAmt'] - prev_amt).fillna(0).astype('float32')
    df['uid_amt_ratio_prev'] = (
        df['TransactionAmt'] / prev_amt.replace(0, np.nan)
    ).fillna(1.0).clip(0, 100).astype('float32')

    df['uid_amt_cummean'] = g['TransactionAmt'].transform(
        lambda s: s.shift(1).expanding().mean()
    ).fillna(-1).astype('float32')

    df['uid_amt_cummax'] = g['TransactionAmt'].transform(
        lambda s: s.shift(1).expanding().max()
    ).fillna(-1).astype('float32')

    return df

X_all = add_time_gap_features(X_all)
feature_cols += [
    'delta_seconds_prev', 'uid_count_so_far',
    'uid_prev_amt', 'uid_amt_diff_prev', 'uid_amt_ratio_prev',
    'uid_amt_cummean', 'uid_amt_cummax'
    # , 'delta_log_prev'
]
print('feature columns now:', len(feature_cols))

# how many test rows actually benefit from the bridge?
X_all = X_all.sort_values(['uid', 'DT'], kind='mergesort')
prev_source = X_all.groupby('uid', sort=False)['__source__'].shift(1)
bridge_mask = (X_all['__source__'] == 'test') & (prev_source == 'train')
n_bridge = bridge_mask.sum()
test_total = (X_all['__source__'] == 'test').sum()
print(f'test rows whose immediate previous-UID-transaction is in TRAIN: '
      f'{n_bridge:,} ({n_bridge / test_total:.1%} of test rows)')


In [ ]:
print('X_all.index name :', X_all.index.name)             # should be 'TransactionID'
print('X_all.index[:3]  :', X_all.index[:3].tolist())     # should be 3 TransactionID-like ints
print('y_train.index[:3]:', y_train.index[:3].tolist())   # should match X_all's index
print('overlap:', X_all.index.isin(y_train.index).sum(),
      'of', len(y_train))                                 # should equal len(y_train) = 590540

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

In [ ]:
cols_to_show = [
    "TransactionAmt",
    'uid_prev_amt', 'uid_amt_cummean', 'uid_amt_diff_prev', 'uid_amt_ratio_prev', 'uid_amt_cummax', "card1",
    "D1n", "uid", "uid_FE", 'uid_count_so_far', 'TransactionDT',  'delta_seconds_prev', "day", "D3", "D3n", "dist1", "P_emaildomain",
]

cols = [c for c in cols_to_show if c in X_all.columns]
missing = [c for c in cols_to_show if c not in X_all.columns]

if missing:
    print("Missing columns:", missing)

X_all.loc[X_all["uid"] == 48158, cols].head(20)

In [ ]:
cols_to_show = [
    "DT", "DT_M", "DT_W", "DT_D",
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday",
    "DT_M_total", "DT_W_total", "DT_D_total"
]

# keep only existing columns
cols = [c for c in cols_to_show if c in X_all.columns]
missing = [c for c in cols_to_show if c not in X_all.columns]

if missing:
    print("Missing columns:", missing)

X_all[cols].head(20)

In [ ]:
X_all['uid'].value_counts()

## 4. Imputation + standardization (fit on train rows only)

In [ ]:
RAW_CATEGORICALS = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
RAW_CATEGORICALS = [c for c in RAW_CATEGORICALS if c in X_all.columns]
print(RAW_CATEGORICALS)

In [ ]:
fe_cols = [c for c in X_train.columns if c.endswith("_FE")]

print("Number of _FE columns:", len(fe_cols))
print(fe_cols)

In [ ]:
from collections import Counter

dupe_feature_cols = [c for c, n in Counter(feature_cols).items() if n > 1]
dupe_xall_cols = X_all.columns[X_all.columns.duplicated()].tolist()

print("duplicate feature_cols:", dupe_feature_cols)
print("duplicate X_all columns:", dupe_xall_cols)
feature_cols = list(dict.fromkeys(feature_cols))

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

# ENCODING CATEGORICAL COLUMNS WITH THEIR FREQUENCIES

In [ ]:
def cardinality(col):
    return X_all[col].nunique(dropna=False)

HIGH_CARD = [c for c in RAW_CATEGORICALS if cardinality(c) > 1000]   # FE only
MED_CARD  = [c for c in RAW_CATEGORICALS if 10 <= cardinality(c) <= 1000]  # FE + factorized
LOW_CARD  = [c for c in RAW_CATEGORICALS if cardinality(c) < 10]    # one-hot

In [ ]:
print(HIGH_CARD)
print(MED_CARD)
print(LOW_CARD)

In [ ]:
low_card_raw_cats = [
    c for c in RAW_CATEGORICALS
    if c in X_all.columns and X_all[c].nunique(dropna=True) <= 3
]

np.array(low_card_raw_cats)

In [ ]:
X_all.card4.value_counts()

In [ ]:
# FREQUENCY ENCODE TOGETHER
def encode_FE(df, cols):
    for col in cols:
        vc = df[col].value_counts(dropna=True, normalize=False).to_dict()
        vc[-1] = -1
        nm = col+'_FE'
        df[nm] = df[col].map(vc).astype('float32')
        print(nm,', ',end='')

In [ ]:
FE_RAW_CATS = [c for c in RAW_CATEGORICALS if c not in low_card_raw_cats]
encode_FE(X_all, FE_RAW_CATS)

In [ ]:
X_all.drop(columns=FE_RAW_CATS, inplace=True)

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

In [ ]:
replace_raw_cats = [c for c in FE_RAW_CATS if c in feature_cols]

feature_cols = [
    f"{c}_FE" if c in replace_raw_cats else c
    for c in feature_cols
]

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

In [ ]:
feature_cols = list(dict.fromkeys(feature_cols))

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

In [ ]:
def add_cyclical_features(df, drop_originals=True):
    """Replace cyclical integer columns with sin/cos pairs in [-1, 1]."""
    cyclical = {
        'DT_hour':       24,
        'DT_day_week':   7,
        'DT_day_month':  31,
        'DT_week_month': 5,
    }
    for col, period in cyclical.items():
        if col not in df.columns:
            continue
        rad = 2 * np.pi * df[col].astype('float32') / period
        df[f'{col}_sin'] = np.sin(rad).astype('float32')
        df[f'{col}_cos'] = np.cos(rad).astype('float32')
    if drop_originals:
        df = df.drop(columns=[c for c in cyclical if c in df.columns])
    return df

In [ ]:
cols_to_show = [
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday"
]

# keep only existing columns
cols = [c for c in cols_to_show if c in X_all.columns]
missing = [c for c in cols_to_show if c not in X_all.columns]

if missing:
    print("Missing columns:", missing)

X_all[cols].head(20)

In [ ]:
X_all = add_cyclical_features(X_all)

# Update feature_cols accordingly
for c in ['DT_hour', 'DT_day_week', 'DT_day_month', 'DT_week_month']:
    if c in feature_cols:
        feature_cols.remove(c)
    feature_cols += [f'{c}_sin', f'{c}_cos']


In [ ]:
sin_cos_cols = [c for c in X_all.columns if c.endswith(("_sin", "_cos"))]

print(len(sin_cos_cols))
print(sin_cos_cols)

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

In [ ]:
train_mask = X_all["__source__"].eq("train")

candidate_cols = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(X_all[c])
]

def skew_report(df, cols, sentinel=-1):
    rows = []

    for c in cols:
        s = df[c].replace([np.inf, -np.inf], np.nan).dropna()

        # Ignore sentinel when judging skew
        s_valid = s[s != sentinel]

        if len(s_valid) < 100:
            continue

        rows.append({
            "col": c,
            "min": s_valid.min(),
            "p50": s_valid.quantile(0.50),
            "p95": s_valid.quantile(0.95),
            "p99": s_valid.quantile(0.99),
            "max": s_valid.max(),
            "skew": s_valid.skew(),
            "zero_rate": (s_valid == 0).mean(),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("skew", ascending=False)
        .reset_index(drop=True)
    )


In [ ]:
report = skew_report(X_all.loc[train_mask], candidate_cols)
report

In [ ]:
log1p_candidate_cols = (
    report
    .loc[
        (report["skew"] > 2) &
        (report["min"] >= 0),
        "col"
    ]
    .tolist()
)

np.array(log1p_candidate_cols)

In [ ]:
print(len(log1p_candidate_cols))

In [ ]:
fe_cols = [c for c in feature_cols if c.endswith("_FE")]

print("Number of _FE columns:", len(fe_cols))
print(fe_cols)

In [ ]:
log1p_cols = [
    c for c in log1p_candidate_cols
    if c in X_all.columns
    and c not in sin_cos_cols
    and X_all[c].nunique(dropna=True) > 3
]

np.array(log1p_cols)

In [ ]:
print(len(log1p_cols))

In [ ]:
def signed_log1p(x):
    """For features that can go negative (uid_amt_diff_prev)."""
    return np.sign(x) * np.log1p(np.abs(x))

In [ ]:
def safe_log1p_with_sentinel(s, sentinel=-1):
    out = s.astype('float32').copy()
    valid = s.ne(sentinel)

    if (s.loc[valid] < 0).any():
        raise ValueError("Non-sentinel negative values found before log1p transform")

    out.loc[valid] = np.log1p(s.loc[valid]).astype('float32')
    return out

In [ ]:
cols_to_log1p = list(dict.fromkeys(log1p_cols))
print(np.array(cols_to_log1p))
print(len(cols_to_log1p))

In [ ]:
def has_real_negatives(s, sentinel=-1):
    """True if column has negatives other than the sentinel."""
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    s_valid = s[s != sentinel]
    return (s_valid < 0).any()

negatives = [c for c in cols_to_log1p if has_real_negatives(X_all[c])]
print(f'Columns with non-sentinel negatives: {len(negatives)}')
print(negatives)

In [ ]:
for c in cols_to_log1p:
    X_all[c] = safe_log1p_with_sentinel(X_all[c])

In [ ]:
def signed_log1p_candidates(df, cols, sentinel=-1, min_negatives=100, min_skew=2):
    out = []
    for c in cols:
        s = df[c].replace([np.inf, -np.inf], np.nan).dropna()
        s_valid = s[s != sentinel]
        n_neg = (s_valid < 0).sum()
        n_pos = (s_valid > 0).sum()
        if n_neg < min_negatives or n_pos < min_negatives:
            continue
        if abs(s_valid.skew()) < min_skew and abs(s_valid.kurt()) < 5:
            continue
        out.append({'col': c, 'n_neg': int(n_neg), 'n_pos': int(n_pos),
                    'min': s_valid.min(), 'max': s_valid.max(),
                    'skew': s_valid.skew()})
    return pd.DataFrame(out).sort_values('skew', key=abs, ascending=False)

print(signed_log1p_candidates(X_all, feature_cols))


In [ ]:
X_all['uid_amt_diff_prev'] = signed_log1p(X_all['uid_amt_diff_prev']).astype('float32')

In [ ]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

In [ ]:
print(X_all.isna().sum()[X_all.isna().sum() > 0])

In [ ]:
# X_all[feature_cols] = X_all[feature_cols].fillna(-1).astype('float32')

In [ ]:
train_mask = (X_all['__source__'] == 'train')
scaler = StandardScaler()

binary_cols = [
    c for c in feature_cols
    if c in X_all.columns and X_all[c].nunique(dropna=True) <= 3
]

to_scale = [
    c for c in feature_cols
    if c not in binary_cols
    and c not in sin_cos_cols
]

In [ ]:
np.array(binary_cols)

In [ ]:
np.array(sin_cos_cols)

In [ ]:
print(to_scale)
print(len(to_scale))

In [ ]:
print(feature_cols)
print(len(feature_cols))

In [ ]:
print(X_all[to_scale].dtypes[X_all[to_scale].dtypes != "float32"])

In [ ]:
print('uid_amt_diff_prev range:', X_all['uid_amt_diff_prev'].min(),
                                  X_all['uid_amt_diff_prev'].max())
# After ONLY signed_log1p (before this cell ran): roughly [-12, 12]
# If you see anything like ±200, signed_log1p never ran — that's the original bug

In [ ]:
# 1. The 5 first features should be roughly N(0, 1) — but slightly tighter due to ±5 clip
sub = X_all.loc[train_mask, to_scale]
print('before-scale stats over to_scale:')
print('  min  :', float(sub.min().min()))   # should be exactly -5.0 (or close)
print('  max  :', float(sub.max().max()))   # should be exactly +5.0 (or close)
print('  mean :', float(sub.mean().mean())) # should be ~0
print('  std  :', float(sub.std().mean()))  # should be ~0.7 - 0.95

# 2. The 33 NOT-scaled columns should still live in their natural ranges
not_scaled = [c for c in feature_cols if c not in to_scale]
sub2 = X_all[not_scaled]
print('\nnot_scaled columns range:')
print('  min :', float(sub2.min().min()))   # expect -1 (TF_NULL sentinel) or -1 (cyclic)
print('  max :', float(sub2.max().max()))   # expect +1 (TF_NULL or cyclic) or near 1

# 3. No NaN/Inf anywhere
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
assert np.isfinite(flat).all(), 'NaN/Inf still present'
print('\n✓ all finite')


In [ ]:
feature_cols = list(dict.fromkeys(feature_cols))
to_scale = list(dict.fromkeys(to_scale))

X_all[feature_cols] = (
    X_all[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(-1)
    .astype("float32")
)

train_mask = X_all["__source__"].eq("train")
scaler = StandardScaler()

X_all.loc[train_mask, to_scale] = scaler.fit_transform(
    X_all.loc[train_mask, to_scale]
).astype("float32")

X_all.loc[~train_mask, to_scale] = scaler.transform(
    X_all.loc[~train_mask, to_scale]
).astype("float32")

X_all[to_scale] = X_all[to_scale].clip(lower=-5.0, upper=5.0).astype("float32")

print('train post-scale mean (first 5):',
      X_all.loc[train_mask, feature_cols[:5]].mean().round(3).to_list())
print('train post-scale std  (first 5):',
      X_all.loc[train_mask, feature_cols[:5]].std().round(3).to_list())

In [ ]:
print('uid_amt_diff_prev range:', X_all['uid_amt_diff_prev'].min(),
                                  X_all['uid_amt_diff_prev'].max())
# After signed_log1p AND scaling+clip, should be in [-5, 5]
# If you see anything like ±200, signed_log1p never ran — that's the original bug

In [ ]:
# 1. The 5 first features should be roughly N(0, 1) — but slightly tighter due to ±5 clip
sub = X_all.loc[train_mask, to_scale]
print('post-scale stats over to_scale:')
print('  min  :', float(sub.min().min()))   # should be exactly -5.0 (or close)
print('  max  :', float(sub.max().max()))   # should be exactly +5.0 (or close)
print('  mean :', float(sub.mean().mean())) # should be ~0
print('  std  :', float(sub.std().mean()))  # should be ~0.7 - 0.95

# 2. The 33 NOT-scaled columns should still live in their natural ranges
not_scaled = [c for c in feature_cols if c not in to_scale]
sub2 = X_all[not_scaled]
print('\nnot_scaled columns range:')
print('  min :', float(sub2.min().min()))   # expect -1 (TF_NULL sentinel) or -1 (cyclic)
print('  max :', float(sub2.max().max()))   # expect +1 (TF_NULL or cyclic) or near 1

# 3. No NaN/Inf anywhere
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
assert np.isfinite(flat).all(), 'NaN/Inf still present'
print('\n✓ all finite')

In [ ]:
feature_cols[:5]

In [ ]:
[(c, c in to_scale) for c in feature_cols[:5]]

In [ ]:
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
print('min:', flat.min(),  '   should be ≥ -5')
print('max:', flat.max(),  '   should be ≤ +5')
assert np.isfinite(flat).all()

In [ ]:
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
print('min :', flat.min(),  '  expected ~ -5')
print('max :', flat.max(),  '  expected ~ +5')
print('mean:', flat.mean(), '  expected ~ 0')
print('std :', flat.std(),  '  expected ~ 0.6 - 1.0')
assert np.isfinite(flat).all(), 'still has NaN/Inf'

## 5. Build per-UID windows on the combined frame

## FOR APPROACH SPLITTING FOLDS BY MONTHS

In [ ]:
import importlib
import buidling_windows_folds_by_months
importlib.reload(buidling_windows_folds_by_months)

from buidling_windows_folds_by_months import build_uid_windows


In [ ]:
BUILD_TEST_SEQUENCE = False
SEQUENCE_DTYPE = np.float16
DROP_PREPROCESSED_FRAMES_AFTER_WINDOW = True


def build_uid_windows_dtype(df, feature_cols, window, uid_col='uid', time_col='DT', dtype=np.float16):
    df_sorted = df.sort_values([uid_col, time_col], kind='mergesort')
    feat = df_sorted[feature_cols].to_numpy(dtype=np.float32)
    uids = df_sorted[uid_col].to_numpy()
    orig = df_sorted.index.to_numpy()

    n = len(df_sorted)
    f = len(feature_cols)
    X = np.zeros((n, window, f), dtype=dtype)
    L = np.zeros(n, dtype=np.int64)

    boundaries = np.flatnonzero(np.concatenate([[True], uids[1:] != uids[:-1]]))
    boundaries = np.append(boundaries, n)
    for b_start, b_end in zip(boundaries[:-1], boundaries[1:]):
        block = feat[b_start:b_end]
        for t in range(b_end - b_start):
            start = max(0, t - window + 1)
            seq = block[start:t + 1].astype(dtype, copy=False)
            X[b_start + t, -seq.shape[0]:] = seq
            L[b_start + t] = seq.shape[0]
    return X, orig, L


print('Building UID-aggregation train windows...')
t0 = time.time()
train_frame = X_all.loc[X_all['__source__'].eq('train')].copy()
X_train_seq, train_order, L_train = build_uid_windows_dtype(
    train_frame, feature_cols, WINDOW, dtype=SEQUENCE_DTYPE
)
y_aligned = y_train.loc[train_order].to_numpy(dtype=np.float32)
dt_m_aligned = train_frame.loc[train_order, 'DT_M'].to_numpy()

if BUILD_TEST_SEQUENCE:
    print('Building test windows as requested...')
    X_all_seq, all_order, L_all = build_uid_windows_dtype(
        X_all, feature_cols, WINDOW, dtype=SEQUENCE_DTYPE
    )
    source_in_order = X_all.loc[all_order, '__source__'].to_numpy()
    train_pos = (source_in_order == 'train')
    test_pos = (source_in_order == 'test')
    X_test_seq = X_all_seq[test_pos]
    L_test = L_all[test_pos]
    test_order = all_order[test_pos]
else:
    print('Skipping test sequence build. This ablation only runs validation folds.')
    X_test_seq = np.zeros((0, WINDOW, X_train_seq.shape[2]), dtype=SEQUENCE_DTYPE)
    L_test = np.zeros(0, dtype=np.int64)
    test_order = np.array([], dtype=train_order.dtype)
    train_pos = np.ones(len(train_order), dtype=bool)
    test_pos = np.zeros(0, dtype=bool)

print(f'X_train_seq: {X_train_seq.shape} {X_train_seq.dtype}')
print(f'X_test_seq : {X_test_seq.shape} {X_test_seq.dtype}')
print(f'feature count: {X_train_seq.shape[2]}')
print(f'mean real-steps train: {L_train.mean():.2f}, '
      f'pct with >1 step: {(L_train>1).mean():.1%}, '
      f'pct with ==WINDOW: {(L_train==WINDOW).mean():.1%}')
print(f'build time: {time.time()-t0:.1f}s')

if DROP_PREPROCESSED_FRAMES_AFTER_WINDOW:
    del train_frame
    try:
        del X_train, X_test, X_all
    except NameError:
        pass
    gc.collect()
    print('Dropped preprocessed DataFrames to free memory before model training.')


In [ ]:
# Optional: save the UID-aggregation sequence cache for reuse.
SAVE_UID_AGG_SEQUENCE_CACHE = False
UID_AGG_NPY_DIR = Path("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data_uidagg_npy")

if SAVE_UID_AGG_SEQUENCE_CACHE:
    UID_AGG_NPY_DIR.mkdir(parents=True, exist_ok=True)
    np.save(UID_AGG_NPY_DIR / "X_train_seq.npy", X_train_seq)
    np.save(UID_AGG_NPY_DIR / "L_train.npy", L_train)
    np.save(UID_AGG_NPY_DIR / "train_order.npy", train_order)
    np.save(UID_AGG_NPY_DIR / "y_aligned.npy", y_aligned)
    np.save(UID_AGG_NPY_DIR / "dt_m_aligned.npy", dt_m_aligned)
    np.save(UID_AGG_NPY_DIR / "feature_cols.npy", np.array(feature_cols, dtype=object))
    print(f"Saved UID aggregation sequence cache to {UID_AGG_NPY_DIR}")
else:
    print("Not saving sequence cache. Set SAVE_UID_AGG_SEQUENCE_CACHE=True if needed.")


In [ ]:
# import os, numpy as np
#
# SRC_NPZ = "/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data.npz"
# DST_DIR = "/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data_npy"
# os.makedirs(DST_DIR, exist_ok=True)
#
# data = np.load(SRC_NPZ)                            # slow ONE last time
# for name in data.files:                            # ['X_train_seq', 'L_train', ...]
#     arr = data[name]
#     # force float32 / int64 to match what your DataLoader will cast to
#     if arr.dtype == np.float64:
#         arr = arr.astype(np.float32)
#     np.save(os.path.join(DST_DIR, f"{name}.npy"), arr)
#     print(f"  saved {name:15s} shape={arr.shape}  dtype={arr.dtype}  "
#           f"size={arr.nbytes/1e6:.1f} MB")
#
# print("done — you can delete the old .npz if you want")

In [ ]:
# import os
# import numpy as np
#
# SRC_NPZ = "/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data.npz"
# DST_DIR = "/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data_npy"
# os.makedirs(DST_DIR, exist_ok=True)
#
# data = np.load(SRC_NPZ)
#
# for name in data.files:
#     arr = data[name]
#
#     # Only reduce the big model input arrays
#     if name in ["X_train_seq", "X_test_seq"]:
#         arr = arr.astype(np.float16)
#
#     # Keep other float arrays safer
#     elif arr.dtype == np.float64:
#         arr = arr.astype(np.float32)
#
#     np.save(os.path.join(DST_DIR, f"{name}.npy"), arr)
#
#     print(
#         f"saved {name:15s} shape={arr.shape} dtype={arr.dtype} "
#         f"size={arr.nbytes / 1e6:.1f} MB"
#     )
#
# print("done")

In [ ]:
# print('X_all_seq min/max  :', X_all_seq.min(),  X_all_seq.max())     # expect [-5, 5]
# print('X_train_seq min/max:', X_train_seq.min(), X_train_seq.max())  # expect [-5, 5]
# print('X_test_seq  min/max:', X_test_seq.min(),  X_test_seq.max())   # expect [-5, 5]
# print('any NaN/Inf in X_train_seq:', not np.isfinite(X_train_seq).all())
# print('positive rate in y_aligned:', y_aligned.mean().round(4))      # expect ~0.035
# print('train rows :', X_train_seq.shape[0])
# print('test  rows :', X_test_seq.shape[0])

In [ ]:
# X_train_seq[1]

In [ ]:
# X_test_seq[5000]

In [ ]:
# y_aligned

In [ ]:
# def bad_seq_cols(X_seq, feature_cols):
#     rows = []
#     for i, c in enumerate(feature_cols):
#         x = X_seq[:, :, i]
#         n_nan = np.isnan(x).sum()
#         n_inf = np.isinf(x).sum()
#         if n_nan or n_inf:
#             rows.append((c, int(n_nan), int(n_inf)))
#     return pd.DataFrame(rows, columns=["col", "n_nan", "n_inf"])
#
# bad_seq_cols(X_train_seq, feature_cols).head(50)

In [ ]:
# bad_xall_cols = []
#
# for c in feature_cols:
#     x = X_all[c].to_numpy()
#     if not np.isfinite(x).all():
#         bad_xall_cols.append(c)
#
# bad_xall_cols

In [ ]:
# print('X_train_seq stats:')
# print('  min   :', X_train_seq.min())
# print('  max   :', X_train_seq.max())
# print('  mean  :', X_train_seq.mean())
# print('  std   :', X_train_seq.std())
# print('  any nan:', np.isnan(X_train_seq).any())
# print('  any inf:', np.isinf(X_train_seq).any())

In [ ]:
# flat = X_train_seq.reshape(-1, X_train_seq.shape[-1])  # (n_rows*window, n_features)
# diag = pd.DataFrame({
#     'col':    feature_cols,
#     'min':    flat.min(axis=0),
#     'max':    flat.max(axis=0),
#     'mean':   flat.mean(axis=0),
#     'std':    flat.std(axis=0),
# })
# diag['abs_max'] = np.maximum(np.abs(diag['min']), np.abs(diag['max']))
# print(diag.sort_values('abs_max', ascending=False).head(20))

## 6. Rolling model ablations with UID aggregation sequences

The sequence tensor above keeps UID aggregation columns from `X_train_copy5.parquet`. The cells below run three rolling temporal ablations in one Run All pass: LSTM, temporal CNN + ResNet + Attention, and feature-axis CNN + LSTM.

In [ ]:
import os, gc, math, time, warnings, copy
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

try:
    from tqdm.auto import tqdm
except Exception:
    class _NoTqdm:
        def __init__(self, iterable=None, **kwargs):
            self.iterable = iterable
        def __iter__(self):
            return iter(self.iterable)
        def set_postfix(self, *args, **kwargs):
            pass
    def tqdm(iterable=None, **kwargs):
        return _NoTqdm(iterable)

# ----- General experiment config -----
WINDOW              = 20
MIN_TRAIN_MONTHS    = 3
BATCH               = 512
EPOCHS              = 30
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 6
SEED                = 42
N_SEEDS             = 3       # keep 3 seeds for comparability with previous CNN runs
USE_POS_WEIGHT      = False
PREDICT_TEST        = False   # tuning mode: save OOF only, skip expensive test inference
PROGRESS_BAR        = True    # show batch-level train/validation progress
PROGRESS_UPDATE_EVERY = 25    # reduce tqdm overhead by updating postfix every N batches

# Set to None for auto, or one of: "cpu", "cuda", "mps".
# Useful on Kaggle when GPU quota is exhausted.
DEVICE_OVERRIDE     = None
CPU_NUM_THREADS     = min(8, os.cpu_count() or 1)

# ----- Model width/depth config -----
CNN_FEATURE_CHANNELS= 32      # channels while convolving over feature axis
CNN_EMBED_DIM       = 128     # per-transaction embedding passed to LSTM/GRU
N_RESBLOCKS         = 2       # keep >=2 for comparability; set 1 for a faster ablation

# Temporal CNN + ResNet + Attention config.
CNN_CHANNELS        = 128
N_ATTN_LAYERS       = 1
N_HEADS             = 4
USE_CLS             = True
USE_MAX_POOL        = True
RNN_HIDDEN          = 128
RNN_LAYERS          = 2
RNN_DROPOUT         = 0.3
RNN_BIDIRECTIONAL   = True    # matches Time_Series_LSTM_left_pad_bidirectional_v3 style
USE_PACKED_RNN      = True    # avoids recurrent compute over padded timesteps
FEATURE_ATTN_HIDDEN = 128
ATTN_DROPOUT        = 0.2
STATIC_HIDDEN       = 64
USE_STATIC_TOWER    = True
USE_MEAN_MAX_POOL   = True

# ----- Optional autoencoder pre-filter for CNN-BiGRU-attention ablation -----
# Disabled by default because filtering high reconstruction error can remove true fraud examples.
AE_FILTER_ENABLE    = False
AE_FILTER_QUANTILE  = 0.995
AE_EPOCHS           = 2
AE_BATCH            = 2048
AE_MAX_FIT_ROWS     = 100_000

MODEL_CASES = [
    {"name": "lstm_bidir_uidagg"},
    {"name": "cnn_temporal_uidagg"},
    {"name": "cnn_feature_lstm_uidagg"},
]

RUN_CASES = [case["name"] for case in MODEL_CASES]
OUTPUT_DIR = Path("uidagg_sequence_model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----- Reproducibility -----
torch.manual_seed(SEED)
np.random.seed(SEED)

# ----- Device selection -----
def resolve_device(override=None):
    if override is not None:
        requested = str(override).lower().strip()
        if requested == 'cuda' and torch.cuda.is_available():
            return torch.device('cuda')
        if requested == 'mps' and torch.backends.mps.is_available():
            return torch.device('mps')
        if requested == 'cpu':
            return torch.device('cpu')
        print(f"Requested DEVICE_OVERRIDE={override!r} is unavailable. Falling back to auto device selection.")

    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = resolve_device(DEVICE_OVERRIDE)
if device.type == 'cpu':
    torch.set_num_threads(CPU_NUM_THREADS)

print(f'PyTorch {torch.__version__}  device={device}')
if device.type == 'cpu':
    print(f'CPU threads: {torch.get_num_threads()}')
print('Cases:', RUN_CASES)
print('PREDICT_TEST:', PREDICT_TEST)

# FraudCNNResAttnV2 is defined inline in the next cell for Kaggle compatibility.

N_FEATURES = X_train_seq.shape[2]
print(f'Using UID-aggregation sequence features: {N_FEATURES}')


In [ ]:
# Self-contained temporal CNN + ResNet + Attention model for Kaggle.
class ResBlock1D(nn.Module):
    def __init__(self, c: int, k: int = 3, drop: float = 0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn1   = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn2   = nn.BatchNorm1d(c)
        self.drop  = nn.Dropout(drop)
        self.act   = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x                                   # SKIP BRANCH
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = out + identity                           # SKIP CONNECTION
        return self.act(out)


# =============================================================================
# Transformer encoder block — one self-attention + FF, both with residual+LN
# =============================================================================
class TransformerBlock(nn.Module):
    def __init__(self, c: int, n_heads: int, drop: float = 0.2):
        super().__init__()
        self.attn  = nn.MultiheadAttention(
            embed_dim=c, num_heads=n_heads, dropout=drop, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(c)
        self.ff    = nn.Sequential(
            nn.Linear(c, c * 2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(c * 2, c),
        )
        self.norm2 = nn.LayerNorm(c)

    def forward(self, h: torch.Tensor, key_padding_mask: torch.Tensor) -> torch.Tensor:
        a, _ = self.attn(h, h, h,
                         key_padding_mask=key_padding_mask,
                         need_weights=False)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ff(h))
        return h


# =============================================================================
# Full v2 model
# =============================================================================
class FraudCNNResAttnV2(nn.Module):
    """
    Args
    ----
    n_features        : number of features per timestep (e.g. 244)
    window            : sequence length T (e.g. 20)
    c_hidden          : channel width of Conv stem / ResNet / attention
    n_resblocks       : number of ResBlock1D in the CNN stack
    n_attn_layers     : number of TransformerBlock layers (was 1 in v1)        ◄── NEW
    n_heads           : self-attention heads (must divide c_hidden)
    drop              : dropout used in ResBlocks, attention, FF, head, static tower
    static_hidden     : hidden width of the static-row tower
    use_cls           : prepend a learnable CLS token, use its embedding for readout ◄── NEW
    use_max_pool      : concat masked-max pool with mean pool                      ◄── NEW
    use_static_tower  : run the last raw row through an MLP and concat it          ◄── NEW
    output_dim        : final logit dim (keep =1 for BCEWithLogitsLoss)
    """
    def __init__(self,
                 n_features: int,
                 window: int = 20,
                 c_hidden: int = 128,
                 n_resblocks: int = 3,
                 n_attn_layers: int = 2,
                 n_heads: int = 4,
                 drop: float = 0.2,
                 static_hidden: int = 64,
                 use_cls: bool = True,
                 use_max_pool: bool = True,
                 use_static_tower: bool = True,
                 output_dim: int = 1):
        super().__init__()
        assert c_hidden % n_heads == 0, "c_hidden must be divisible by n_heads"

        self.window           = window
        self.use_cls          = use_cls
        self.use_max_pool     = use_max_pool
        self.use_static_tower = use_static_tower

        # ------- Stage 1 — Conv1D stem -------
        self.stem = nn.Sequential(
            nn.Conv1d(n_features, c_hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(c_hidden),
            nn.ReLU(),
        )

        # ------- Stage 2 — ResNet1D stack -------
        self.resblocks = nn.Sequential(*[
            ResBlock1D(c_hidden, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])

        # ------- Stage 3 — positional embedding (over the WINDOW positions only;
        #                  the CLS token is added on top after this)
        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden))
        nn.init.trunc_normal_(self.pos, std=0.02)

        # ------- (Optional) CLS token -------
        if use_cls:
            self.cls = nn.Parameter(torch.zeros(1, 1, c_hidden))
            nn.init.trunc_normal_(self.cls, std=0.02)

        # ------- Stage 4 — STACK of Transformer blocks -------
        self.attn_blocks = nn.ModuleList([
            TransformerBlock(c_hidden, n_heads, drop=drop)
            for _ in range(n_attn_layers)
        ])

        # ------- (Optional) Static-row tower -------
        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
                nn.Linear(256, static_hidden), nn.ReLU(),
            )

        # ------- Compute pool dim from the toggles -------
        pool_dim = c_hidden                       # mean is always present
        if use_max_pool:    pool_dim += c_hidden
        if use_cls:         pool_dim += c_hidden
        if use_static_tower:pool_dim += static_hidden

        # ------- Stage 5 — Head -------
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(pool_dim, 64), nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, output_dim),
        )

    # ----------------------------------------------------------------------
    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)  left-padded     lengths: (B,) real-step count
        B, T, F = x.shape

        # Keep last (most-recent) raw row aside for the static tower.
        # Left-padding writes to the RIGHT end, so x[:, -1, :] is always real.
        last_row = x[:, -1, :]                           # (B, F)

        # ----- CNN path (B, T, F) → (B, T, C) -----
        x_t = x.transpose(1, 2)                          # (B, F, T)
        h   = self.stem(x_t)                             # (B, C, T)
        h   = self.resblocks(h)                          # (B, C, T)
        h   = h.transpose(1, 2)                          # (B, T, C)
        h   = h + self.pos                               # positional encoding (over T only)

        # ----- Build real-position mask from `lengths` -----
        idx          = torch.arange(T, device=h.device).unsqueeze(0)   # (1, T)
        pos_from_end = T - 1 - idx                                     # (1, T)
        real         = pos_from_end < lengths.unsqueeze(1)             # (B, T) — True = real

        # ----- (Optional) Prepend CLS token -----
        if self.use_cls:
            cls = self.cls.expand(B, -1, -1)             # (B, 1, C)
            h   = torch.cat([cls, h], dim=1)             # (B, T+1, C)
            # CLS is always "real" so attention won't mask it
            cls_real = torch.ones(B, 1, dtype=torch.bool, device=h.device)
            real_full = torch.cat([cls_real, real], dim=1)              # (B, T+1)
            key_padding_mask = ~real_full
        else:
            key_padding_mask = ~real

        # ----- Stacked Transformer blocks -----
        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)

        # ----- Split CLS embedding from the time tokens -----
        if self.use_cls:
            cls_out = h[:, 0, :]                         # (B, C)
            seq_h   = h[:, 1:, :]                        # (B, T, C)
        else:
            seq_h = h                                    # (B, T, C)

        # ----- Pool over REAL timesteps only -----
        mask_f = real.unsqueeze(-1).float()              # (B, T, 1)
        cnt    = mask_f.sum(dim=1).clamp(min=1.0)        # (B, 1)

        pooled_parts = [(seq_h * mask_f).sum(dim=1) / cnt]   # mean

        if self.use_max_pool:
            # mask out padded positions with -inf so they never win the max
            seq_h_for_max = seq_h.masked_fill(~real.unsqueeze(-1), float('-inf'))
            pooled_parts.append(seq_h_for_max.max(dim=1).values)

        if self.use_cls:
            pooled_parts.append(cls_out)

        if self.use_static_tower:
            pooled_parts.append(self.static_mlp(last_row))

        pooled = torch.cat(pooled_parts, dim=1)          # (B, pool_dim)
        return self.head(pooled)                         # (B, 1)


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = torch.from_numpy(X)
        self.lengths = torch.from_numpy(lengths.astype('int64'))
        self.y = None if y is None else torch.from_numpy(y.astype('float32'))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        if self.y is None:
            return self.X[i], self.lengths[i]
        return self.X[i], self.lengths[i], self.y[i]


def make_loader(X, lengths, y, batch_size, shuffle):
    return DataLoader(
        WindowDataset(X, lengths, y),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )


In [ ]:
class LSTMBidirectionalClassifier(nn.Module):
    def __init__(self, n_features, hidden_dim=RNN_HIDDEN, num_layers=RNN_LAYERS,
                 drop_prob=RNN_DROPOUT, output_dim=1, use_static_tower=True,
                 bidirectional=True):
        super().__init__()
        self.use_static_tower = use_static_tower
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)
        seq_out_dim = 2 * lstm_out_dim
        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(128, 64), nn.ReLU(),
            )
            combined_dim = seq_out_dim + 64
        else:
            combined_dim = seq_out_dim
        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, output_dim),
        )

    def forward(self, x, lengths):
        B, T, _ = x.shape
        lstm_out, _ = self.lstm(x)
        idx = torch.arange(T, device=x.device).unsqueeze(0)
        pos_from_end = T - 1 - idx
        mask = pos_from_end < lengths.to(x.device).unsqueeze(1)
        mask_f = mask.unsqueeze(-1).float()
        mean_pool = (lstm_out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
        neg_inf = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values
        seq_vec = torch.cat([mean_pool, max_pool], dim=1)
        if self.use_static_tower:
            stat_vec = self.static_mlp(x[:, -1, :])
            seq_vec = torch.cat([seq_vec, stat_vec], dim=1)
        return self.head(seq_vec).squeeze(-1)


def left_padded_real_mask(lengths: torch.Tensor, T: int, device=None) -> torch.Tensor:
    """Return True for real timesteps in a left-padded sequence."""
    if device is None:
        device = lengths.device
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    idx = torch.arange(T, device=device).unsqueeze(0)
    pos_from_end = T - 1 - idx
    return pos_from_end < lengths.unsqueeze(1)


def right_padded_real_mask(lengths: torch.Tensor, T: int, device=None) -> torch.Tensor:
    """Return True for real timesteps in a right-padded sequence."""
    if device is None:
        device = lengths.device
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    idx = torch.arange(T, device=device).unsqueeze(0)
    return idx < lengths.unsqueeze(1)


def left_to_right_padded(h: torch.Tensor, lengths: torch.Tensor):
    """
    Convert left-padded sequence embeddings to right-padded embeddings.

    pack_padded_sequence expects real timesteps to start at index 0. The sequence cache is
    left-padded, so real timesteps are at the end of the window and must be shifted left
    before packed LSTM/GRU execution.
    """
    B, T, E = h.shape
    device = h.device
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    dst_pos = torch.arange(T, device=device).unsqueeze(0).expand(B, T)
    src_pos = (T - lengths.unsqueeze(1) + dst_pos).clamp(min=0, max=T - 1)
    out = h.gather(1, src_pos.unsqueeze(-1).expand(B, T, E))
    real = dst_pos < lengths.unsqueeze(1)
    out = out.masked_fill(~real.unsqueeze(-1), 0.0)
    return out, real


def masked_mean_max(h: torch.Tensor, real: torch.Tensor):
    """Masked mean and max over temporal dimension."""
    mask_f = real.unsqueeze(-1).float()
    mean = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
    neg_inf = torch.finfo(h.dtype).min
    maxv = h.masked_fill(~real.unsqueeze(-1), neg_inf).max(dim=1).values
    return mean, maxv


class FeatureAttentionGate(nn.Module):
    """
    Feature-wise attention gate.

    Input : x with shape (B, T, F)
    Output: reweighted x and feature weights with shape (B, T, F)

    This attention operates over input features inside each transaction row, not over timesteps.
    Padding timesteps are skipped instead of being processed by the MLP.
    """
    def __init__(self, n_features, hidden=FEATURE_ATTN_HIDDEN, drop=ATTN_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden, n_features),
            nn.Sigmoid(),
        )

    def forward(self, x, real=None):
        B, T, F = x.shape
        flat = x.reshape(B * T, F)

        if real is None:
            weights = self.net(flat).view(B, T, F)
            return x * weights, weights

        real_idx = real.reshape(-1).nonzero(as_tuple=False).squeeze(1)
        weights_flat = flat.new_zeros(B * T, F)
        if real_idx.numel() > 0:
            weights_real = self.net(flat.index_select(0, real_idx))
            weights_flat = weights_flat.index_copy(0, real_idx, weights_real)
        weights = weights_flat.view(B, T, F)
        return x * weights, weights


class FeatureResBlock1D(nn.Module):
    """Residual Conv1D block over the feature axis of a single transaction row."""
    def __init__(self, c: int, k: int = 3, drop: float = 0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn1   = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn2   = nn.BatchNorm1d(c)
        self.drop  = nn.Dropout(drop)
        self.act   = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        return self.act(out + identity)


class CNNFeatureExtractor(nn.Module):
    """
    CNN local feature extractor.

    Input : (B, T, F)
    Step  : select only real timesteps, reshape to (N_real, 1, F),
            so Conv1D slides along feature dimension F.
    Output: (B, T, embed_dim), one feature-interaction embedding per real transaction.
    """
    def __init__(self, n_features, feature_channels=CNN_FEATURE_CHANNELS,
                 embed_dim=CNN_EMBED_DIM, n_resblocks=N_RESBLOCKS, drop=ATTN_DROPOUT):
        super().__init__()
        self.n_features = n_features
        self.embed_dim = embed_dim
        self.stem = nn.Sequential(
            nn.Conv1d(1, feature_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(feature_channels),
            nn.ReLU(),
        )
        self.resblocks = nn.Sequential(*[
            FeatureResBlock1D(feature_channels, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])
        self.proj = nn.Sequential(
            nn.Linear(feature_channels * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x, real=None):
        B, T, F = x.shape
        flat = x.reshape(B * T, F)

        if real is None:
            row_x = flat.unsqueeze(1).contiguous()       # (B*T, 1, F)
            h = self.stem(row_x)
            h = self.resblocks(h)
            mean_pool = h.mean(dim=2)
            max_pool = h.max(dim=2).values
            return self.proj(torch.cat([mean_pool, max_pool], dim=1)).view(B, T, self.embed_dim)

        real_idx = real.reshape(-1).nonzero(as_tuple=False).squeeze(1)
        row_emb_flat = flat.new_zeros(B * T, self.embed_dim)
        if real_idx.numel() == 0:
            return row_emb_flat.view(B, T, self.embed_dim)

        row_x = flat.index_select(0, real_idx).unsqueeze(1).contiguous()  # (N_real, 1, F)
        h = self.stem(row_x)                                             # (N_real, C, F)
        h = self.resblocks(h)                                            # (N_real, C, F)
        mean_pool = h.mean(dim=2)                                        # (N_real, C)
        max_pool = h.max(dim=2).values                                   # (N_real, C)
        row_emb = self.proj(torch.cat([mean_pool, max_pool], dim=1))      # (N_real, E)
        row_emb_flat = row_emb_flat.index_copy(0, real_idx, row_emb)
        return row_emb_flat.view(B, T, self.embed_dim)


class StaticTower(nn.Module):
    def __init__(self, n_features, static_hidden=STATIC_HIDDEN, drop=RNN_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, static_hidden), nn.ReLU(),
        )

    def forward(self, last_row):
        return self.net(last_row)


class CNNRNNHybridClassifier(nn.Module):
    """
    Hybrid model where feature attention weights critical input features,
    CNN extracts local feature interactions per transaction row,
    then LSTM/GRU models the temporal sequence across rows.
    """
    def __init__(self, n_features, rnn_type='lstm', use_feature_attention=False,
                 cnn_embed_dim=CNN_EMBED_DIM, rnn_hidden=RNN_HIDDEN, rnn_layers=RNN_LAYERS,
                 bidirectional=RNN_BIDIRECTIONAL, drop=RNN_DROPOUT,
                 use_static_tower=USE_STATIC_TOWER, use_mean_max_pool=USE_MEAN_MAX_POOL,
                 use_packed_rnn=USE_PACKED_RNN):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        self.use_feature_attention = use_feature_attention
        self.use_static_tower = use_static_tower
        self.use_mean_max_pool = use_mean_max_pool
        self.use_packed_rnn = use_packed_rnn

        if use_feature_attention:
            self.feature_attention = FeatureAttentionGate(n_features)

        self.feature_cnn = CNNFeatureExtractor(n_features, embed_dim=cnn_embed_dim)

        rnn_cls = nn.GRU if self.rnn_type == 'gru' else nn.LSTM
        self.rnn = rnn_cls(
            input_size=cnn_embed_dim,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            dropout=drop if rnn_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        rnn_out_dim = rnn_hidden * (2 if bidirectional else 1)

        pool_dim = rnn_out_dim
        if use_mean_max_pool:
            pool_dim += 2 * rnn_out_dim

        if use_static_tower:
            self.static_mlp = StaticTower(n_features, static_hidden=STATIC_HIDDEN, drop=drop)
            pool_dim += STATIC_HIDDEN

        self.head = nn.Sequential(
            nn.Linear(pool_dim, 64), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        B, T, _ = x.shape
        lengths = lengths.to(device=x.device, dtype=torch.long).clamp(min=1, max=T)
        left_real = left_padded_real_mask(lengths, T, device=x.device)

        if self.use_feature_attention:
            x_used, feature_weights = self.feature_attention(x, real=left_real)
        else:
            x_used = x

        # For left-padded windows, the current transaction is always at index T-1.
        # For attention cases, the static tower sees the feature-weighted current row.
        last_row = x_used[:, -1, :]

        # CNN extracts local feature interactions only for real transaction rows.
        h_left = self.feature_cnn(x_used, real=left_real)  # (B, T, CNN_EMBED_DIM), left-padded

        if self.use_packed_rnn:
            h, rnn_real = left_to_right_padded(h_left, lengths)
            packed = pack_padded_sequence(
                h,
                lengths.detach().cpu(),
                batch_first=True,
                enforce_sorted=False,
            )
            packed_out, _ = self.rnn(packed)
            rnn_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)
            rnn_out = rnn_out.masked_fill(~rnn_real.unsqueeze(-1), 0.0)
            last_idx = (lengths - 1).clamp(min=0)
            last_hidden = rnn_out[torch.arange(B, device=x.device), last_idx, :]
        else:
            rnn_out, _ = self.rnn(h_left)
            rnn_real = left_real
            rnn_out = rnn_out.masked_fill(~rnn_real.unsqueeze(-1), 0.0)
            last_hidden = rnn_out[:, -1, :]

        parts = [last_hidden]

        if self.use_mean_max_pool:
            mean, maxv = masked_mean_max(rnn_out, rnn_real)
            parts.extend([mean, maxv])

        if self.use_static_tower:
            parts.append(self.static_mlp(last_row))

        feat = torch.cat(parts, dim=1)
        return self.head(feat).squeeze(-1)


def make_model(case, n_features):
    kind = case['kind']
    if kind == 'cnn_lstm':
        return CNNRNNHybridClassifier(n_features, rnn_type='lstm', use_feature_attention=False)
    if kind == 'cnn_bigru_attention':
        return CNNRNNHybridClassifier(n_features, rnn_type='gru', use_feature_attention=True, bidirectional=True)
    if kind == 'cnn_lstm_attention':
        return CNNRNNHybridClassifier(n_features, rnn_type='lstm', use_feature_attention=True)
    raise ValueError(f"Unknown model kind: {kind}")


def make_model(case, n_features):
    if case['name'] == 'lstm_bidir_uidagg':
        return LSTMBidirectionalClassifier(n_features, use_static_tower=True)
    if case['name'] == 'cnn_temporal_uidagg':
        return FraudCNNResAttnV2(
            n_features,
            c_hidden=CNN_CHANNELS,
            n_resblocks=N_RESBLOCKS,
            n_attn_layers=N_ATTN_LAYERS,
            n_heads=N_HEADS,
            drop=ATTN_DROPOUT,
            static_hidden=STATIC_HIDDEN,
            use_cls=USE_CLS,
            use_max_pool=USE_MAX_POOL,
            use_static_tower=USE_STATIC_TOWER,
        )
    if case['name'] == 'cnn_feature_lstm_uidagg':
        return CNNRNNHybridClassifier(
            n_features,
            rnn_type='lstm',
            use_feature_attention=False,
        )
    raise ValueError(f"Unknown model case: {case}")


MODEL_CASES = [
    {"name": "lstm_bidir_uidagg"},
    {"name": "cnn_temporal_uidagg"},
    {"name": "cnn_feature_lstm_uidagg"},
]
RUN_CASES = [case["name"] for case in MODEL_CASES]
print("Model cases:", RUN_CASES)


In [ ]:
def expanding_month_folds(months_array, min_train_months=MIN_TRAIN_MONTHS):
    months = sorted(np.unique(months_array).tolist())
    for vm in months[min_train_months:]:
        tm = [m for m in months if m < vm]
        ti = np.flatnonzero(np.isin(months_array, tm))
        vi = np.flatnonzero(months_array == vm)
        yield (vm, tm, ti, vi)


def train_one_fold(case, X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    if case.get('use_ae_filter', False) and AE_FILTER_ENABLE:
        print('      fitting autoencoder filter...', flush=True)
        keep = fit_autoencoder_filter(X_tr, L_tr, n_features, device)
        X_tr, L_tr, y_tr = X_tr[keep], L_tr[keep], y_tr[keep]
        print(f"      after AE filter: train rows={len(X_tr):,}, positive rate={y_tr.mean():.4f}", flush=True)

    model = make_model(case, n_features).to(device)

    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum())
        neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        print(
            f"   ep {epoch:>2}/{epochs} start "
            f"(train_batches={len(train_loader):,}, valid_batches={len(val_loader):,})",
            flush=True,
        )

        train_iter = tqdm(
            train_loader,
            total=len(train_loader),
            desc=f"{case['name']} seed{seed} ep{epoch}/{epochs} train",
            leave=False,
            dynamic_ncols=True,
            disable=not PROGRESS_BAR,
        )
        for step_idx, (xb, lb, yb) in enumerate(train_iter, start=1):
            xb = xb.to(device=device, dtype=torch.float32)
            lb = lb.to(device=device, dtype=torch.long)
            yb = yb.to(device=device, dtype=torch.float32)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb, lb).squeeze(-1)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()

            running += loss.item() * xb.size(0)
            n_seen += xb.size(0)
            if step_idx == 1 or step_idx % PROGRESS_UPDATE_EVERY == 0 or step_idx == len(train_loader):
                current_lr = scheduler.get_last_lr()[0]
                train_iter.set_postfix(loss=f"{running / max(n_seen, 1):.4f}", lr=f"{current_lr:.2e}")
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        val_iter = tqdm(
            val_loader,
            total=len(val_loader),
            desc=f"{case['name']} seed{seed} ep{epoch}/{epochs} valid",
            leave=False,
            dynamic_ncols=True,
            disable=not PROGRESS_BAR,
        )
        with torch.no_grad():
            for xb, lb, _ in val_iter:
                xb = xb.to(device=device, dtype=torch.float32)
                lb = lb.to(device=device, dtype=torch.long)
                logits = model(xb, lb).squeeze(-1)
                preds.append(torch.sigmoid(logits).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f"   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)", flush=True)
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds
            bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f"   early stop at epoch {epoch}", flush=True)
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val_preds, best_auc, model


In [ ]:
fold_specs = list(expanding_month_folds(dt_m_aligned, MIN_TRAIN_MONTHS))
print(f"{len(fold_specs)} folds; {N_SEEDS} seeds per fold", flush=True)
print(f"Test prediction enabled: {PREDICT_TEST}", flush=True)

case_lookup = {case['name']: case for case in MODEL_CASES}
experiment_summaries = []


def clear_device_cache():
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    elif device.type == 'mps':
        torch.mps.empty_cache()
    gc.collect()


def predict_test_for_model(model):
    test_loader = make_loader(X_test_seq, L_test, y=None, batch_size=BATCH, shuffle=False)
    tps = []
    model.eval()
    test_iter = tqdm(
        test_loader,
        total=len(test_loader),
        desc="test inference",
        leave=False,
        dynamic_ncols=True,
        disable=not PROGRESS_BAR,
    )
    with torch.no_grad():
        for xb, lb in test_iter:
            xb = xb.to(device=device, dtype=torch.float32)
            lb = lb.to(device=device, dtype=torch.long)
            logits = model(xb, lb).squeeze(-1)
            tps.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(tps)


def run_experiment_case(case_name, predict_test=PREDICT_TEST):
    if case_name not in case_lookup:
        raise ValueError(f"Unknown case_name={case_name!r}. Available: {list(case_lookup)}")

    case = case_lookup[case_name]
    print("\n" + "=" * 90, flush=True)
    print(f"RUNNING CASE: {case_name}", flush=True)
    print(f"predict_test={predict_test}", flush=True)
    print("=" * 90, flush=True)

    oof = np.full(len(X_train_seq), np.nan, dtype=np.float32)
    test_preds = np.zeros(len(X_test_seq), dtype=np.float32) if predict_test and len(X_test_seq) else None
    fold_aucs = []

    for fold, (vm, tm, idxT, idxV) in enumerate(fold_specs):
        print(f"\n=== {case_name} | Fold {fold}: train {tm} -> validate {vm} "
              f"(train={len(idxT):,}, valid={len(idxV):,}) ===")

        seed_val_preds = []
        seed_test_preds = [] if predict_test else None

        for s in range(N_SEEDS):
            seed = SEED + s
            print(f"-- seed {seed} --", flush=True)
            vp, va, model = train_one_fold(
                case,
                X_train_seq[idxT], L_train[idxT], y_aligned[idxT],
                X_train_seq[idxV], L_train[idxV], y_aligned[idxV],
                n_features=N_FEATURES, epochs=EPOCHS, batch=BATCH,
                lr=LR, weight_decay=WEIGHT_DECAY, device=device,
                early_stop_patience=EARLY_STOP_PATIENCE,
                grad_clip=GRAD_CLIP, seed=seed,
            )
            seed_val_preds.append(vp)

            if predict_test and len(X_test_seq):
                seed_test_preds.append(predict_test_for_model(model))

            del model
            clear_device_cache()

        fold_val = np.mean(seed_val_preds, axis=0)
        fold_auc = roc_auc_score(y_aligned[idxV], fold_val)
        print(f"   fold AUC (seed-avg) = {fold_auc:.4f}")

        fold_aucs.append((int(vm), float(fold_auc)))
        oof[idxV] = fold_val

        if predict_test and len(X_test_seq):
            test_preds += np.mean(seed_test_preds, axis=0)

    if predict_test and len(fold_specs):
        test_preds /= len(fold_specs)

    validated = ~np.isnan(oof)
    overall_auc = roc_auc_score(y_aligned[validated], oof[validated])
    print(f"\n=== {case_name} OOF AUC (validated months only) = {overall_auc:.4f} ===")
    print(f"   per-fold: {fold_aucs}")

    oof_df = pd.DataFrame({
        'TransactionID': train_order,
        f'oof_{case_name}': oof,
        'isFraud': y_aligned,
    })

    oof_path = OUTPUT_DIR / f"oof_{case_name}.csv"
    oof_df.to_csv(oof_path, index=False)
    print(f"Saved {oof_path}")

    test_path = None
    if predict_test and len(X_test_seq):
        test_df = pd.DataFrame({
            'TransactionID': test_order,
            f'pred_{case_name}': test_preds,
        })
        test_path = OUTPUT_DIR / f"test_pred_{case_name}.csv"
        test_df.to_csv(test_path, index=False)
        print(f"Saved {test_path}")
    else:
        print("Skipped test prediction. Set PREDICT_TEST=True for final submission inference.")

    summary_row = {
        'case': case_name,
        'overall_oof_auc': float(overall_auc),
        'fold_aucs': repr(fold_aucs),
        'oof_path': str(oof_path),
        'test_path': '' if test_path is None else str(test_path),
        'predict_test': bool(predict_test),
        'n_seeds': int(N_SEEDS),
        'n_folds': int(len(fold_specs)),
        'n_resblocks': int(N_RESBLOCKS),
        'skip_padded_cnn_rows': True,
        'use_packed_rnn': bool(USE_PACKED_RNN),
        'ae_filter_enabled': bool(AE_FILTER_ENABLE and case.get('use_ae_filter', False)),
    }

    summary_path = OUTPUT_DIR / f"summary_{case_name}.csv"
    pd.DataFrame([summary_row]).to_csv(summary_path, index=False)
    print(f"Saved {summary_path}")

    experiment_summaries.append(summary_row)
    return summary_row


In [ ]:
# Run all UID-aggregation ablations in one pass.
uidagg_experiment_summaries = []
for case_name in RUN_CASES:
    uidagg_experiment_summaries.append(run_experiment_case(case_name, predict_test=False))

uidagg_summary_df = pd.DataFrame(uidagg_experiment_summaries)
uidagg_summary_path = OUTPUT_DIR / "summary_all_uidagg_sequence_models.csv"
uidagg_summary_df.to_csv(uidagg_summary_path, index=False)
print(f"Saved {uidagg_summary_path}")
uidagg_summary_df
